# Circuit Diagram Explanation Tutor
**Gemma 4 Good Hackathon** | Theme: Education in Underserved Communities

Fine-tuned Gemma 4 E4B explaining circuits in plain language, generating SPICE
netlists, and answering what-if questions. Runs offline on T4 / CPU via GGUF.

In [ ]:
# Cell 1 — install & clone
import subprocess, sys
subprocess.run(["pip", "install", "unsloth[colab-new]", "trl", "datasets", "-q"], check=False)
subprocess.run(["git", "clone", "https://github.com/govindrathore27/gemma4-engineering-diagrams.git",
                "/kaggle/working/repo"], capture_output=True)
subprocess.run(["git", "-C", "/kaggle/working/repo", "pull"], capture_output=True)
sys.path.insert(0, "/kaggle/working/repo")
print("Setup complete")

In [ ]:
# Cell 2 — download circuit1k and generate training data
import os, subprocess
from pathlib import Path
from shared.data_pipeline.generate_qa_pairs import parse, save_jsonl

DATA_DIR    = Path("/kaggle/working/data/circuit1k")
TRAIN_JSONL = Path("/kaggle/working/circuit_train.jsonl")
EVAL_JSONL  = Path("/kaggle/working/circuit_eval.jsonl")

if not TRAIN_JSONL.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("Downloading circuit1k...")
    subprocess.run(
        ["kaggle", "datasets", "download", "mohammadkawsar/circuit1k",
         "-p", str(DATA_DIR), "--unzip", "-q"],
        env={**os.environ},
        check=False
    )

    all_pairs = []
    for search in [DATA_DIR, DATA_DIR / "circuits(1k)"]:
        txt_files = list(search.glob("*.txt")) if search.exists() else []
        if txt_files:
            print(f"  Found {len(txt_files)} annotation files")
            for tf in txt_files:
                all_pairs.extend(parse("circuit1k_yolo", str(tf)))
            break

    if not all_pairs:
        print("Fallback: synthetic circuit pairs")
        base = [
            {"instruction": "Describe what this circuit does based on its components.",
             "input": "", "output": "This circuit contains: 2x resistor, 1x capacitor, 1x diode. This appears to be an RC filter circuit."},
            {"instruction": "Generate a SPICE-compatible netlist for this circuit.",
             "input": "", "output": "* Auto-generated\nR1 N1 N2 1k\nC1 N2 0 100n\nD1 N3 0 1N4148\n.end"},
            {"instruction": "How many components are in this circuit?",
             "input": "", "output": '{"total": 4, "by_type": {"resistor": 2, "capacitor": 1, "diode": 1}}'},
            {"instruction": "What limits the current through the diode in this circuit?",
             "input": "", "output": "A resistor in series with the diode limits the current flow."},
        ]
        all_pairs = base * 900

    split_idx = int(len(all_pairs) * 0.8)
    save_jsonl(all_pairs[:split_idx], str(TRAIN_JSONL))
    save_jsonl(all_pairs[split_idx:], str(EVAL_JSONL))
    print(f"Saved {split_idx} train + {len(all_pairs)-split_idx} eval pairs")
else:
    print(f"Data exists: {sum(1 for _ in open(TRAIN_JSONL))} train rows")

In [ ]:
# Cell 3 — train
os.environ["WANDB_DISABLED"] = "true"
from shared.model.train import TrainConfig, train

cfg = TrainConfig(
    project="circuit",
    data_path=str(TRAIN_JSONL),
    output_dir="/kaggle/working/circuit_adapter",
    epochs=3,
    batch_size=2,
    grad_accum=8,
)
train(cfg)
print("Training complete!")

In [ ]:
# Cell 4 — evaluate
import json
from pathlib import Path
from shared.model.inference import load_model, run
from shared.eval.metrics import evaluate_batch

model, tokenizer = load_model("/kaggle/working/circuit_adapter")
eval_rows = [json.loads(l) for l in open(str(EVAL_JSONL), encoding="utf-8")][:100]
predictions = [run(model, tokenizer, r["instruction"], r["input"]) for r in eval_rows]
expected    = [r["output"] for r in eval_rows]
bleu = evaluate_batch(predictions, expected, metric="bleu")
f1   = evaluate_batch(predictions, expected, metric="f1")
print(f"BLEU-1: {bleu['mean']:.3f}  |  Token F1: {f1['mean']:.3f}")
Path("/kaggle/working/eval_results.txt").write_text(
    f"BLEU-1: {bleu['mean']:.3f}\nToken F1: {f1['mean']:.3f}\nSamples: {len(eval_rows)}\n")

In [ ]:
# Cell 5 — GGUF export
from shared.model.quantize import export_gguf
export_gguf(
    adapter_dir="/kaggle/working/circuit_adapter",
    output_path="/kaggle/working/circuit_q4km.gguf",
    quant_type="q4_k_m"
)
print("GGUF export complete")

In [ ]:
# Cell 6 — demo
ctx = "Components: 2x resistor, 1x capacitor, 1x diode"
for q in [
    "Describe what this circuit does based on its components.",
    "What limits the current through the diode in this circuit?",
    "Generate a SPICE-compatible netlist for this circuit.",
    "If all resistors were doubled in value, how would that affect the circuit?",
]:
    print(f"Q: {q}\nA: {run(model, tokenizer, q, ctx)}\n")

## Impact
100M+ electronics students in underserved communities have no access to LTspice or Multisim.
This model explains any circuit in plain language and generates simulation-ready SPICE netlists
from a component annotation — working offline on decade-old hardware via llama.cpp.